# Laya V2.9 — Mathematical Audit Before Any More CeNN Training

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V29_Mathematical_Audit_Colab.ipynb)

**Purpose:** Test whether the present attention-free MultiscaleCeNN can represent key full-attention behaviors **before** spending GPU time on another 400–1600 optimizer steps. This notebook performs **no training**.

The existing V2.9 run reported layer-15 audit cosine **0.82079**, decision agreement **0.834**, teacher gold accuracy **0.844**, student gold accuracy **0.736**, and layer latency **8.692ms versus 1.404ms**. These are user-provided GPU results, not measurements reproduced by this notebook.

This audit includes:
1. Exact recurrence/pooled-state receptive-field calculation.
2. A numerical replay of the teacher's native attention formula, checking correct QKV, RoPE, additive attention mask, and output projection.
3. Actual teacher attention-mass locality and remote key selectivity at radii 32/47/64/96/128/159.
4. An **optimistic diagnostic proxy**: *exact teacher-derived near attention* plus *uniform far-value mean weighted by the true far mass*. It is **not an upper or lower bound** on any nonlinear CeNN family.
5. A controlled swap: exchange two distant input tokens within an aligned pair. Original/adaptive CeNN pooled summaries and outputs at an unaffected query are provably invariant, whereas RoPE full attention can distinguish their positions. Reports the **observed** teacher sensitivity; it does not assume every random swap yields a large response.
6. Save a self-contained JSON audit. No official test split is accessed.

**Caution:** A counterexample proves the architectural family cannot represent *every* attention behavior. It does not alone prove that the measured task requires those behaviors. The empirical diagnostics evaluate task relevance.


In [1]:
#@title 1. Install exact training dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2029
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 20 #@param {type:"integer"}
DEV_CASES = 100 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 4 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 8 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 8 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V29_Mathematical_Audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert STEPS == 400 and DEV_CASES >= 4, "Use 400-step screening and a meaningful held-out dev split"
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
print("Math audit only; training: DISABLED; official test used: NO")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

rl_agent_config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

GPU: Tesla T4 AMP: torch.float16
Source: convaiinnovations/laya-typed-decisions
Target: 15 all full attention layers: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]
Math audit only; training: DISABLED; official test used: NO


In [3]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_embeddings=entry["pos"],
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V2.9 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

train/probe/audit decisions: 100 250 250
unique train/probe/audit cases: 20 50 50
official test remains untouched


In [4]:
#@title 4. Attention-free adaptive multiscale cellular encoder
class AdaptiveHierarchicalCeNN(nn.Module):
    """Attention-free token mixer.

    Uses per-scale masked pooling, finite-step contractive depthwise cellular
    recurrence, token-dependent scale gates, and global channelwise moments.
    Absolutely no Q-K score matrices, SDPA, linear attention, or cross-token
    content-based attention; no sequence-dependent learned token pair scores.
    """
    architecture = "adaptive_hierarchical_multiscale_cenn"

    def __init__(self, original, cell_dim=384, steps=3, scales=(1,4,16,64)):
        super().__init__()
        self.config = original.config
        self.cell_dim = int(cell_dim)
        self.steps = int(steps)
        self.scales = tuple(int(s) for s in scales)
        if self.cell_dim < 1 or self.steps < 1 or not self.scales or any(s < 1 for s in self.scales):
            raise ValueError("Invalid adaptive CeNN dimensions")
        d = int(self.config.hidden_size)
        self.down = nn.Linear(d, self.cell_dim, bias=False)
        self.up = nn.Linear(self.cell_dim, d, bias=False)
        self.norm = nn.LayerNorm(self.cell_dim)
        self.template = nn.Parameter(torch.zeros(self.cell_dim, 1, 5))
        self.leak_logit = nn.Parameter(torch.zeros(self.cell_dim))
        self.input_gain = nn.Parameter(torch.ones(self.cell_dim))
        self.global_proj = nn.Linear(2 * self.cell_dim, self.cell_dim)
        self.global_gain = nn.Parameter(torch.full((self.cell_dim,), 0.1))
        self.route = nn.Linear(self.cell_dim, len(self.scales))
        self.route_global = nn.Linear(self.cell_dim, len(self.scales), bias=False)
        self.route_temperature = nn.Parameter(torch.tensor(0.0))
        self.readout_gain = nn.Parameter(torch.tensor(0.0))
        self.direct_gate = nn.Parameter(torch.tensor(-1.0))
        self.out_drop = copy.deepcopy(original.out_drop)
        # Initialize cell read/write paths using pretrained VALUE+output maps.
        self.value_proj = nn.Linear(d, d, bias=original.Wqkv.bias is not None)
        self.value_out = copy.deepcopy(original.Wo)
        with torch.no_grad():
            self.value_proj.weight.copy_(original.Wqkv.weight[2*d:3*d])
            if self.value_proj.bias is not None:
                self.value_proj.bias.copy_(original.Wqkv.bias[2*d:3*d])
            pick = min(self.cell_dim, d)
            self.down.weight[:pick].copy_(self.value_proj.weight[:pick])
            self.up.weight[:, :pick].copy_(original.Wo.weight[:, :pick])
            nn.init.zeros_(self.template)
            self.template[:, 0, 2] = 0.05
            nn.init.zeros_(self.route.weight)
            nn.init.zeros_(self.route.bias)
            nn.init.zeros_(self.route_global.weight)
            nn.init.normal_(self.global_proj.weight, std=0.002)
            nn.init.zeros_(self.global_proj.bias)
        self.value_proj.requires_grad_(False)
        self.value_out.requires_grad_(False)

    def forward(self, hidden_states, attention_mask=None, **kwargs):
        # No RoPE needed: recurrence and multiscale pools encode locality/order.
        b, t, _ = hidden_states.shape
        if attention_mask is None:
            valid = torch.ones((b, t), device=hidden_states.device, dtype=torch.bool)
        elif attention_mask.ndim == 2:
            valid = attention_mask.bool()
        else:
            # Allow standard ModernBERT full-attention 4-D additive/bool masks.
            from tinycenn_lm.laya_lab.core import _valid_tokens
            valid = _valid_tokens(attention_mask, hidden_states)
        m = valid[:, :, None].to(hidden_states.dtype)
        u = self.norm(self.down(hidden_states)) * m
        counts = m.sum(1).clamp_min(1)
        gmean = u.sum(1) / counts
        grms = torch.sqrt((u.square() * m).sum(1) / counts + 1e-6)
        glob = self.global_proj(torch.cat([gmean, grms], dim=-1))
        context = (glob * self.global_gain[None, :])[:, :, None]
        kernels = torch.tanh(self.template.float())
        kernels = 0.90 * kernels / kernels.abs().sum(-1, keepdim=True).clamp_min(1.0)
        leak = torch.sigmoid(self.leak_logit.float())[None, :, None]
        branches = []
        for scale in self.scales:
            pad = (-t) % scale
            upad = F.pad(u.transpose(1,2), (0, pad))
            mpad = F.pad(m.transpose(1,2), (0, pad))
            n = (t + pad) // scale
            cnt = mpad.reshape(b, 1, n, scale).sum(-1).clamp_min(1)
            pooled = upad.reshape(b, self.cell_dim, n, scale).sum(-1) / cnt
            pmask = (mpad.reshape(b, 1, n, scale).sum(-1) > 0).to(u.dtype)
            drive = pooled * self.input_gain[None, :, None] + context
            state = torch.tanh(pooled) * pmask
            for _ in range(self.steps):
                feedback = F.conv1d(state, kernels.to(state.dtype), padding=2, groups=self.cell_dim)
                state = ((1 - leak) * state + leak * torch.tanh(feedback + drive)) * pmask
            branches.append(state.repeat_interleave(scale, dim=-1)[..., :t].transpose(1, 2))
        route_input = self.route(u) + self.route_global(glob)[:, None, :]
        routes = torch.softmax(route_input / self.route_temperature.clamp(-1,1).exp(), dim=-1)
        stack = torch.stack(branches, dim=2)  # B,T,scales,C
        mixed = (stack * routes[..., None]).sum(2) * m
        local = self.up(mixed) * self.readout_gain.clamp(-2,2).exp()
        direct = self.value_out(self.value_proj(hidden_states)) * m
        gate = self.direct_gate.clamp(-5,5).sigmoid()
        out = ((1-gate)*local + gate*direct) * m
        return self.out_drop(out), None

    def config_dict(self):
        return dict(architecture=self.architecture, cell_dim=self.cell_dim,
                    steps=self.steps, scales=list(self.scales),
                    attention_free=True, global_context="mean_rms",
                    token_routing="per_token_multiscale_gates",
                    value_only_pretrained_anchor=True)

CANDIDATE_INFO = {
    "cenn_v25": dict(label="Original MultiscaleCeNN 256x2", cell_dim=256,
                     steps=2, scales=(1,4,16), lr=(0.01,0.003)),
    "cenn_large": dict(label="Large MultiscaleCeNN 512x4", cell_dim=512,
                       steps=4, scales=(1,2,4,8,16,32), lr=(0.008,0.002)),
    "adaptive_384": dict(label="Adaptive Hierarchical CeNN 384x3", cell_dim=384,
                         steps=3, scales=(1,4,16,64), lr=(0.003,0.0006)),
    "adaptive_512": dict(label="Adaptive Hierarchical CeNN 512x4", cell_dim=512,
                         steps=4, scales=(1,2,8,32,128), lr=(0.003,0.0006)),
}
CANDIDATE_NAMES = tuple(CANDIDATE_INFO)

def build_candidate(name, original):
    recipe = CANDIDATE_INFO[name]
    if name.startswith("adaptive_"):
        model = AdaptiveHierarchicalCeNN(
            original, cell_dim=recipe["cell_dim"],
            steps=recipe["steps"], scales=recipe["scales"],
        )
    else:
        model = MultiscaleCeNN(
            original, cell_dim=recipe["cell_dim"],
            steps=recipe["steps"], scales=recipe["scales"],
        )
    return model.to(device)

def candidate_parameters(module):
    return [p for p in module.parameters() if p.requires_grad]

print("Candidates (ALL attention-free):")
for name, info in CANDIDATE_INFO.items():
    print(" ", name, "->", info["label"])


Candidates (ALL attention-free):
  cenn_v25 -> Original MultiscaleCeNN 256x2
  cenn_large -> Large MultiscaleCeNN 512x4
  adaptive_384 -> Adaptive Hierarchical CeNN 384x3
  adaptive_512 -> Adaptive Hierarchical CeNN 512x4


## A. Mathematical structure and what is fundamentally lost

Let \(u_j=\operatorname{LN}(W_dx_j)\). At a scale \(s\), the present implementation first pools

\[
p_{s,k}=\frac{1}{|B_{s,k}|}\sum_{j\in B_{s,k}}u_j,
\qquad g = G\left(\frac{1}{T}\sum_j u_j\right).
\]

It then applies \(K\) local cellular updates on the **pooled** sequence, broadcasts the result to tokens in each bin, and combines scales. The adaptive variant adds a per-query gate \(\pi_{i,s}\) computed from \(u_i\) and global channel moments. A gate over scales is **not** key-based content-addressable retrieval.

For kernel radius \(r=(w_{\mathrm{kernel}}-1)/2\), each update extends coarse-grid influence by at most \(r\) cells. With \(K\) updates and scale \(s\), the conservative maximum input-token influence radius through this *scale branch* is

\[
R_s\leq(s-1)+Krs.
\]

This excludes the global-summary path, which is everywhere but is not individually addressable. A branch can have a large structural reach while losing information through pooling.

**Exact invariance:** Choose two distant positions \(j,j+1\) with even \(j\), outside the fine-scale recurrent neighborhood of query \(i=0\). Swap their hidden-state vectors, leaving all other tokens fixed. Since they remain in the same aligned bin for **every** scale \(s>1\) in the tested configurations, every pooled vector at those scales is identical. Their per-token features are merely permuted, so global mean (and RMS for adaptive) are also identical. At scale \(s=1\), finite-step recurrence cannot transmit the swap to the distant query. Thus its output at query 0 is identical up to floating-point roundoff. Full attention with RoPE generally need not be identical, because the key vectors receive different positional rotations after the swap.

This is an architecture-level identifiability limitation **for the current mean-pooled CeNN family**. Increasing cell width or training steps cannot remove this exact pooling invariance; only changing what information is retained/retrieved can.

**Stability versus expressivity:** The original recurrence normalizes the depthwise cellular kernel to an absolute-sum at most 0.9. For a fixed pooled input and gate \(0<\eta<1\), the state update obeys the channelwise bound

\[
\left\|\frac{\partial h^{(r+1)}}{\partial h^{(r)}}\right\|_\infty
\le (1-\eta)+0.9\eta=1-0.1\eta<1.
\]

Thus the cellular state update is contractive for fixed input. This explains stability, **not** its ability to invert pooling or selectively retrieve arbitrary distant keys. Repeating the contractive update cannot re-create information discarded before the recurrence.

The original CeNN also has approximately scale-invariant token features because LayerNorm follows \(W_dx\); query-sensitive changes in key/value magnitude may therefore be poorly represented. This is another hypothesis to test, not a proof of failure on real model states.


In [5]:
#@title 5. Static receptive-field mathematics (no training)
def mathematical_reach(name):
    cfg = CANDIDATE_INFO[name]
    r = 2 if name.startswith("adaptive_") else 1
    K = cfg["steps"]
    scales = cfg["scales"]
    return {
        "architecture": name,
        "kernel_radius": r,
        "cell_steps": K,
        "scales": list(scales),
        "branch_max_radii": {str(s): (s - 1) + K * r * s for s in scales},
        "maximum_structural_reach": max((s - 1) + K * r * s for s in scales),
        "distant_key_addressability": False,
        "global_route": "mean+RMS channel moments" if name.startswith("adaptive_") else "global mean",
    }

static_results = [mathematical_reach(name) for name in CANDIDATE_NAMES]
for row in static_results:
    print(f"{row['architecture']:<17} reach <= {row['maximum_structural_reach']:>5} tokens "
          f"| per-scale radii: {row['branch_max_radii']}")
print("These bounds refer to the pooled cellular paths, not global summary broadcasting.")
print("Original TRAIN_CACHE_BATCHES=48 and BATCH=3 used only 144 distinct cached decisions per layer.")


cenn_v25          reach <=    47 tokens | per-scale radii: {'1': 2, '4': 11, '16': 47}
cenn_large        reach <=   159 tokens | per-scale radii: {'1': 4, '2': 9, '4': 19, '8': 39, '16': 79, '32': 159}
adaptive_384      reach <=   447 tokens | per-scale radii: {'1': 6, '4': 27, '16': 111, '64': 447}
adaptive_512      reach <=  1151 tokens | per-scale radii: {'1': 8, '2': 17, '8': 71, '32': 287, '128': 1151}
These bounds refer to the pooled cellular paths, not global summary broadcasting.
Original TRAIN_CACHE_BATCHES=48 and BATCH=3 used only 144 distinct cached decisions per layer.


In [6]:
#@title 6. Replay the exact pretrained ModernBERT attention equation, inspect locality
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb

def manual_teacher_attention(entry):
    module = teacher.encoder.layers[LAYER].attn
    impl = module.config._attn_implementation
    if impl not in ("sdpa", "eager"):
        raise RuntimeError(f"Manual attention audit requires SDPA/eager, got {impl}")
    x = entry["x"]
    b, t, hidden = x.shape
    h, d = module.num_heads, module.head_dim
    kw = entry["call_kwargs"]
    position_ids = kw.get("position_ids")
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=amp_dtype):
        qkv = module.Wqkv(x).view(b, t, 3, h, d)
        cos, sin = module.rotary_emb(qkv, position_ids=position_ids)
        q, k, v = qkv.transpose(3, 1).unbind(dim=2)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        mask = kw.get("attention_mask")
        if module.local_attention != (-1, -1):
            raise RuntimeError("Mathematical audit target is not a full-attention layer")
        core = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
        exact_reconstruction = module.out_drop(
            module.Wo(core.transpose(1, 2).contiguous().view(b, t, hidden))
        )
    valid = entry["valid"]
    error = exact_reconstruction.float()[valid] - entry["y"].float()[valid]
    truth = entry["y"].float()[valid]
    reconstruction_nmse = float(error.square().mean()/truth.square().mean().clamp_min(1e-8))
    reconstruction_cos = float(F.cosine_similarity(
        exact_reconstruction.float()[valid], entry["y"].float()[valid], dim=-1
    ).mean())
    if reconstruction_nmse > .02:
        raise RuntimeError(
            f"Attention reconstruction mismatched model output (NMSE={reconstruction_nmse:.5f}). "
            "Do not interpret the attention statistics before fixing RoPE/masks."
        )

    # Explicit FP32 attention probabilities are for *diagnostics only*.
    # They are never computed by a CeNN replacement at inference.
    scores = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(d)
    if mask is not None:
        if mask.dtype == torch.bool:
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        else:
            scores = scores + mask.float()
    weights = torch.softmax(scores, dim=-1).nan_to_num(0.0)
    valid_keys = valid[:, None, None, :].to(weights.dtype)
    weights = weights * valid_keys
    weights = weights / weights.sum(-1, keepdim=True).clamp_min(1e-8)
    return dict(q=q.detach(), k=k.detach(), v=v.detach(),
                weights=weights.detach(), valid=valid,
                reconstruction_nmse=reconstruction_nmse,
                reconstruction_cos=reconstruction_cos)

def teacher_attention_statistics(result, radii=(32,47,64,96,128,159)):
    a = result["weights"]
    valid = result["valid"]
    b, h, t, _ = a.shape
    indices = torch.arange(t, device=a.device)
    distances = (indices[:, None] - indices[None, :]).abs()[None, None]
    valid_queries = valid[:, None, :].expand(b, h, t)
    out = {}
    for radius in radii:
        remote = (distances > radius)
        remote_w = a * remote.to(a.dtype)
        mass = remote_w.sum(-1)
        best = remote_w.max(-1).values
        effective = best / mass.clamp_min(1e-8)
        nonzero = valid_queries & (mass > .01)
        out[str(radius)] = dict(
            mean_remote_mass=float(mass[valid_queries].mean()),
            fraction_queries_remote_mass_gt_0_10=float((mass[valid_queries] > .10).float().mean()),
            mean_max_remote_conditional_probability=float(
                effective[nonzero].mean()) if bool(nonzero.any()) else 0.0,
            remote_active_queries=int(nonzero.sum()),
        )
    # Reconstructed teacher weights should sum to one over valid keys.
    row_sums=a.sum(-1)[valid_queries]
    out["normalization_max_abs_error"]=float((row_sums-1).abs().max())
    return out

probe_cache = cache_teacher_io(LAYER, probe_items, PROBE_CACHE_BATCHES)
attention_stats = {str(r): [] for r in (32,47,64,96,128,159)}
attention_replays = []
analysis_entries = []
for index, entry in enumerate(probe_cache[:min(4,len(probe_cache))]):
    reconstruction = manual_teacher_attention(entry)
    stats = teacher_attention_statistics(reconstruction)
    for r in attention_stats:
        attention_stats[r].append(stats[r])
    attention_replays.append(dict(index=index,
                                  cosine=reconstruction["reconstruction_cos"],
                                  nmse=reconstruction["reconstruction_nmse"],
                                  row_normalization_max_error=stats["normalization_max_abs_error"]))
    analysis_entries.append((entry,reconstruction))
    print(f"Batch {index}: manual teacher reconstruction cosine={reconstruction['reconstruction_cos']:.6f} "
          f"NMSE={reconstruction['reconstruction_nmse']:.7f}")
remote_summary = {}
for key, rows in attention_stats.items():
    remote_summary[key] = {
        k: float(np.mean([row[k] for row in rows]))
        for k in ("mean_remote_mass", "fraction_queries_remote_mass_gt_0_10",
                  "mean_max_remote_conditional_probability")
    }
print("\nActual full-attention tail statistics (sample, held-out probe):")
for radius, metrics in remote_summary.items():
    print(f"Outside R={int(radius):3d}: mass={metrics['mean_remote_mass']:.3f}, "
          f"queries with >10% far mass={metrics['fraction_queries_remote_mass_gt_0_10']:.1%}, "
          f"mean largest conditional far key={metrics['mean_max_remote_conditional_probability']:.3f}")
print("\nInterpretation: far mass by itself does not prove content-specific retrieval; "
      "remote concentration and counterfactual response give additional evidence.")


Batch 0: manual teacher reconstruction cosine=1.000000 NMSE=0.0000000
Batch 1: manual teacher reconstruction cosine=1.000000 NMSE=0.0000000
Batch 2: manual teacher reconstruction cosine=1.000000 NMSE=0.0000000
Batch 3: manual teacher reconstruction cosine=1.000000 NMSE=0.0000000

Actual full-attention tail statistics (sample, held-out probe):
Outside R= 32: mass=0.701, queries with >10% far mass=94.7%, mean largest conditional far key=0.225
Outside R= 47: mass=0.619, queries with >10% far mass=92.5%, mean largest conditional far key=0.233
Outside R= 64: mass=0.541, queries with >10% far mass=89.7%, mean largest conditional far key=0.245
Outside R= 96: mass=0.407, queries with >10% far mass=81.5%, mean largest conditional far key=0.274
Outside R=128: mass=0.293, queries with >10% far mass=68.1%, mean largest conditional far key=0.311
Outside R=159: mass=0.187, queries with >10% far mass=48.5%, mean largest conditional far key=0.330

Interpretation: far mass by itself does not prove cont

In [7]:
#@title 7. Optimistic exact-near + uniform-remote oracle proxy (diagnostic, not capacity bound)
def oracle_uniform_far(entry, stats, radii=(47,96,159)):
    module = teacher.encoder.layers[LAYER].attn
    A, v, valid = stats["weights"], stats["v"], stats["valid"]
    b, h, t, d = v.shape
    indices = torch.arange(t, device=v.device)
    dist = (indices[:, None] - indices[None, :]).abs()[None, None]
    key_allowed = valid[:, None, None, :]
    outputs={}
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=amp_dtype):
        for r in radii:
            near = (dist <= r) & key_allowed
            far = (dist > r) & key_allowed
            local_exact = (A * near).to(v.dtype) @ v
            far_mass = (A * far).sum(-1, keepdim=True).to(v.dtype)
            far_count = far.float().sum(-1, keepdim=True).clamp_min(1.0)
            uniform_far = (far.float() / far_count).to(v.dtype) @ v
            core = local_exact + far_mass * uniform_far
            approx = module.out_drop(
                module.Wo(core.transpose(1,2).reshape(b,t,h*d))
            )
            selector = valid
            y = entry["y"].float()[selector]
            z = approx.float()[selector]
            cos=float(F.cosine_similarity(y,z,dim=-1).mean())
            nmse=float(((y-z)**2).mean()/y.square().mean().clamp_min(1e-8))
            outputs[str(r)]=dict(cosine=cos,nmse=nmse)
    return outputs

oracle_batch_results = [oracle_uniform_far(entry, stats) for entry,stats in analysis_entries]
oracle_results={}
for key in ("47","96","159"):
    oracle_results[key] = dict(
        cosine=float(np.mean([row[key]["cosine"] for row in oracle_batch_results])),
        nmse=float(np.mean([row[key]["nmse"] for row in oracle_batch_results]))
    )
print("Oracle PROXY: exact near weighted retrieval, true query-specific far mass, "
      "but only mean far values (stronger than current CeNN in its local branch)")
for key, val in oracle_results.items():
    print(f"Radius {key}: cosine={val['cosine']:.5f} NMSE={val['nmse']:.5f}")
print("These are achievable diagnostic approximations for this proxy; "
      "they are NOT universal bounds on the CeNN architecture.")


Oracle PROXY: exact near weighted retrieval, true query-specific far mass, but only mean far values (stronger than current CeNN in its local branch)
Radius 47: cosine=0.67021 NMSE=0.59879
Radius 96: cosine=0.80049 NMSE=0.38186
Radius 159: cosine=0.92653 NMSE=0.13696
These are achievable diagnostic approximations for this proxy; they are NOT universal bounds on the CeNN architecture.


In [8]:
#@title 8. Exact aligned-pair swap invariance: CeNN output versus RoPE full attention
# Use a held-out hidden-state sample. No gradient and no optimizer.
entry = next((e for e in probe_cache if int(e["valid"].sum(-1).max()) >= 48), None)
if entry is None:
    raise RuntimeError("Need at least one >=48-token example for distant-swap audit")
lengths = entry["valid"].sum(-1)
bidx=int(lengths.argmax().item())
length=int(lengths[bidx].item())
# All even adjacent pairs are in the same bins for scales 2,4,8,16,... .
# Query position 0 is beyond the 4-step scale-1 cellular radius for j>=16.
pairs=list(range(16,max(17,length-1),2))
if len(pairs) > 16:
    pair_ids=np.linspace(0,len(pairs)-1,16,dtype=int).tolist()
    pairs=[pairs[i] for i in sorted(set(pair_ids))]
pairs=[j for j in pairs if j+1 < length]
if not pairs:
    raise RuntimeError("Could not find a remote aligned token pair")
baseline_x=entry["x"]
orig_teacher=teacher.encoder.layers[LAYER].attn
models={n:build_candidate(n,orig_teacher).eval() for n in CANDIDATE_NAMES}
swap_records=[]
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=amp_dtype):
    teacher_base = _attention_output_tensor(
        orig_teacher(*entry["call_args"],**entry["call_kwargs"])
    )[bidx,0].float()
    cenn_base={n:m(baseline_x, attention_mask=entry["mask"],
                   position_embeddings=entry["pos"])[0][bidx,0].float()
               for n,m in models.items()}
    for j in pairs:
        edited=baseline_x.clone()
        edited[bidx,j]=baseline_x[bidx,j+1]
        edited[bidx,j+1]=baseline_x[bidx,j]
        args=(edited,)+tuple(entry["call_args"][1:])
        altered_teacher=_attention_output_tensor(
            orig_teacher(*args,**entry["call_kwargs"])
        )[bidx,0].float()
        # Normalize by baseline output magnitude for interpretable sensitivity.
        teacher_relative=float(
            (altered_teacher-teacher_base).norm()/teacher_base.norm().clamp_min(1e-6)
        )
        differences={}
        for n,m in models.items():
            alt=m(edited,attention_mask=entry["mask"],
                  position_embeddings=entry["pos"])[0][bidx,0].float()
            differences[n]=float(
                (alt-cenn_base[n]).norm()/cenn_base[n].norm().clamp_min(1e-6)
            )
        swap_records.append(dict(j=j, teacher_relative_change=teacher_relative,
                                 cenn_relative_changes=differences))
        print(f"Swap ({j},{j+1}) teacher Δrel={teacher_relative:.7f} | "
              + " ".join(f"{n}={differences[n]:.7f}" for n in CANDIDATE_NAMES))
for m in models.values(): del m
del models

teacher_best=max(swap_records,key=lambda x:x["teacher_relative_change"])
cenn_max={n:max(row["cenn_relative_changes"][n] for row in swap_records)
          for n in CANDIDATE_NAMES}
print("\nLargest observed teacher response:",teacher_best["teacher_relative_change"])
print("Max CeNN response at query 0:", cenn_max)
if any(val > .01 for val in cenn_max.values()):
    print("WARNING: CeNN swap invariance was not numerically tight; inspect masks, dropout or FP accumulation.")
if teacher_best["teacher_relative_change"] < 1e-3:
    print("NOTE: sampled adjacent swaps gave small teacher responses. "
          "This does not invalidate the exact CeNN pooling invariance proof.")


Swap (16,17) teacher Δrel=0.0029899 | cenn_v25=0.0000000 cenn_large=0.0000000 adaptive_384=0.0000000 adaptive_512=0.0000000
Swap (38,39) teacher Δrel=0.0019093 | cenn_v25=0.0000000 cenn_large=0.0000000 adaptive_384=0.0000000 adaptive_512=0.0000000
Swap (60,61) teacher Δrel=0.0025189 | cenn_v25=0.0000000 cenn_large=0.0000000 adaptive_384=0.0000000 adaptive_512=0.0000000
Swap (82,83) teacher Δrel=0.0019400 | cenn_v25=0.0000000 cenn_large=0.0000000 adaptive_384=0.0000000 adaptive_512=0.0000000
Swap (104,105) teacher Δrel=0.0003491 | cenn_v25=0.0000000 cenn_large=0.0000000 adaptive_384=0.0000000 adaptive_512=0.0000000
Swap (126,127) teacher Δrel=0.0006987 | cenn_v25=0.0000000 cenn_large=0.0000000 adaptive_384=0.0000000 adaptive_512=0.0000000
Swap (148,149) teacher Δrel=0.0009935 | cenn_v25=0.0000000 cenn_large=0.0000000 adaptive_384=0.0000000 adaptive_512=0.0000000
Swap (170,171) teacher Δrel=0.0009232 | cenn_v25=0.0000000 cenn_large=0.0000000 adaptive_384=0.0000000 adaptive_512=0.0000000


In [9]:
#@title 9. Save mathematical assessment and explicit STOP/GO questions
report={
    "version": "2.9-math-audit",
    "target_layer": LAYER,
    "model": SOURCE_MODEL,
    "train_updates": 0,
    "official_test_used": False,
    "statement": "Current mean-pooled CeNN lacks arbitrary query-key addressability",
    "static_receptive_field": static_results,
    "teacher_replay": attention_replays,
    "teacher_remote_attention": remote_summary,
    "exact_near_uniform_far_proxy": oracle_results,
    "aligned_pair_swap": {
        "query": 0, "chosen_batch_index": bidx, "sequence_length": length,
        "pairs": swap_records, "max_cenn_relative_deltas": cenn_max,
        "max_teacher_relative_delta": teacher_best["teacher_relative_change"],
    },
    "experiment_caveats": [
        "No training or source model changes",
        "Oracle proxy is not a theorem or capacity upper bound",
        "Swap invariance is a representational counterexample, not proof of task-level impossibility",
        "Teacher attention weights computed explicitly only for diagnostics",
        "Probe subset is sampled, not an official test result",
        "Actual teacher attention uses RoPE and content-addressed keys",
    ],
}
path=OUTPUT_DIR/"v29_mathematical_audit.json"
path.write_text(json.dumps(report,indent=2,allow_nan=False),encoding="utf-8")
print("\nSaved:",path)
print("\nINTERPRETATION")
print("1. Mean pooling + fixed cellular recurrence cannot recover arbitrary key-specific remote tokens.")
print("2. Assess observed teacher far-mass and far-key concentration above.")
print("3. Compare optimistic proxy fidelity against the current audit cosine 0.8208.")
print("4. Train no more identical V2.9 candidates until a new addressing/storage mechanism is specified.")
print("5. A candidate must improve both fidelity and *measured* inference speed; larger CeNN alone failed.")
print("6. If task-specific teacher far selectivity is weak, test cheaper local token mixers instead.")



Saved: /content/Laya_V29_Mathematical_Audit/v29_mathematical_audit.json

INTERPRETATION
1. Mean pooling + fixed cellular recurrence cannot recover arbitrary key-specific remote tokens.
2. Assess observed teacher far-mass and far-key concentration above.
3. Compare optimistic proxy fidelity against the current audit cosine 0.8208.
4. Train no more identical V2.9 candidates until a new addressing/storage mechanism is specified.
5. A candidate must improve both fidelity and *measured* inference speed; larger CeNN alone failed.
6. If task-specific teacher far selectivity is weak, test cheaper local token mixers instead.
